# Day 82: Multi-Modal Request Batching

**Layer:** Modalities


## Concept Overview

A multi-modal server handles a mix of text-only and image+text requests in the same batch. Image requests require the encoder to run first; their image tokens are then concatenated with the text tokens.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Multi-Modal Batch Scheduler

Route image-containing requests to image encoder first, then merge with text-only requests into a continuous batch.


In [ ]:
import numpy as np

class MultiModalBatchScheduler:
    def __init__(self, max_batch=8, img_encoder_slots=4):
        self.text_queue = []
        self.image_queue = []
        self.max_batch = max_batch
        self.img_slots = img_encoder_slots

    def enqueue(self, req):
        if req.get("has_image"):
            self.image_queue.append(req)
        else:
            self.text_queue.append(req)

    def form_batch(self):
        batch = []
        # Process image requests through encoder first
        for r in self.image_queue[:self.img_slots]:
            r["image_tokens"] = 576  # LLaVA-style patch count
            r["total_tokens"] = r.get("text_tokens", 50) + 576
            batch.append(r)
        self.image_queue = self.image_queue[self.img_slots:]
        # Fill remaining slots with text-only
        remaining = self.max_batch - len(batch)
        batch.extend(self.text_queue[:remaining])
        self.text_queue = self.text_queue[remaining:]
        return batch

np.random.seed(42)
sched = MultiModalBatchScheduler()
for i in range(20):
    has_img = np.random.random() < 0.4
    sched.enqueue({"id": i, "has_image": has_img, "text_tokens": np.random.randint(20,200)})

print("Multi-modal batch formation:")
for step in range(3):
    b = sched.form_batch()
    imgs = [r["id"] for r in b if r.get("has_image")]
    txts = [r["id"] for r in b if not r.get("has_image")]
    total_toks = sum(r.get("total_tokens", r.get("text_tokens",50)) for r in b)
    print(f"  Batch {step+1}: {len(b)} reqs | img={imgs} text={txts} | total_tokens={total_toks}")


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- A multi-modal server handles a mix of text-only and image+text requests in the same batch.
- A multi-modal server handles a mix of text-only and image+text requests in the s....
- Day 82 implementation complete.
